# Clasificador perro / gato (Keras)

Clasifica una imagen indicando la clase y el **porcentaje de confianza**.

Instrucciones:
1. Ejecuta la celda de imports/modelo.
2. Ejecuta la celda de la función `clasificar`.
3. Sube una imagen con la celda de carga, o escribe su ruta en `RUTA_IMAGEN`.
4. Ejecuta la última celda: imprime la clase y dibuja el gráfico con Matplotlib.

In [ ]:
import os
import keras
import matplotlib.pyplot as plt

IMAGE_SIZE = (180, 180)

candidatos = [
    '/content/cats_dogs_model.keras',
    'cats_dogs_model.keras',
    'code/cats_dogs_model.keras',
]
ruta_modelo = next((r for r in candidatos if os.path.exists(r)), None)
if ruta_modelo is None:
    raise FileNotFoundError('Sube cats_dogs_model.keras a /content/')
modelo = keras.models.load_model(ruta_modelo)
print('Modelo cargado:', ruta_modelo)

In [ ]:
def clasificar(ruta_imagen):
    img = keras.utils.load_img(ruta_imagen, target_size=IMAGE_SIZE)
    arr = keras.utils.img_to_array(img)
    arr = keras.ops.expand_dims(arr, 0)
    logit = modelo.predict(arr, verbose=0)[0][0]
    p_perro = float(keras.ops.sigmoid(logit))
    p_gato = 1.0 - p_perro
    if p_perro >= 0.5:
        return 'perro', p_perro * 100, p_gato * 100, p_perro * 100
    return 'gato', p_gato * 100, p_gato * 100, p_perro * 100

## Subir imagen (opcional)

En Colab puedes subir la imagen con este widget; también puedes copiarla directamente a `/content/`.

In [ ]:
try:
    from google.colab import files
    subidos = files.upload()
    RUTA_IMAGEN = next(iter(subidos)) if subidos else None
    print('Imagen subida:', RUTA_IMAGEN)
except Exception as exc:
    print('Carga interactiva no disponible:', exc)
    RUTA_IMAGEN = None

In [ ]:
# Si no subiste una imagen, escribe aquí su ruta (por ejemplo /content/mi_imagen.jpg).
if RUTA_IMAGEN is None:
    RUTA_IMAGEN = '/content/mi_imagen.jpg'

etiqueta, confianza, p_gato, p_perro = clasificar(RUTA_IMAGEN)
print(f'{RUTA_IMAGEN} -> {etiqueta} (confianza: {confianza:.2f}%)')
print(f'  gato: {p_gato:.2f}% | perro: {p_perro:.2f}%')

In [ ]:
# Gráfico de probabilidades con Matplotlib
etiquetas = ["Gato", "Perro"]
valores = [p_gato, p_perro]
colores = ["#4C78A8", "#F58518"]
fig, ax = plt.subplots(figsize=(6, 4))
barras = ax.bar(etiquetas, valores, color=colores)
ax.set_ylabel("Probabilidad (%)")
ax.set_ylim(0, 100)
ax.set_title("Clasificación perro / gato")
ax.grid(axis="y", alpha=0.3)
for barra, valor in zip(barras, valores):
    ax.text(barra.get_x() + barra.get_width() / 2, valor + 1.5, f"{valor:.2f}%", ha="center")
fig.tight_layout()
plt.show()
